In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/Telco-Customer-Churn.csv")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)

Dataset shape: (7043, 21)


In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing TotalCharges:", df["TotalCharges"].isnull().sum())
print("Data type:", df["TotalCharges"].dtype)

Missing TotalCharges: 11
Data type: float64


In [3]:
X = df.drop(columns=["Churn", "customerID"])
y=df["Churn"].map({"No":0,"Yes":1})

print("Features shape:", X.shape)
print("Target shape: ",y.shape)
print("\nTarget distrubution :\n",y.value_counts())

Features shape: (7043, 19)
Target shape:  (7043,)

Target distrubution :
 Churn
0    5174
1    1869
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training:", X_train.shape, "| Testing:", X_test.shape)

Training: (5634, 19) | Testing: (1409, 19)


In [5]:
numerical_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
categorical_cols.append("SeniorCitizen")

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)
print("Total features:", len(numerical_cols) + len(categorical_cols))

Numerical columns: ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'SeniorCitizen']
Total features: 19


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scaler", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("encoder", OneHotEncoder(handle_unknown="ignore"))])

In [7]:
preprocessor = ColumnTransformer([
    ("num", num_pipe, numerical_cols),
    ("cat", cat_pipe, categorical_cols)
])

print("Preprocessing pipeline created successfully.")
print("Numerical columns:", len(numerical_cols))
print("Categorical columns:", len(categorical_cols))

Preprocessing pipeline created successfully.
Numerical columns: 3
Categorical columns: 16


In [8]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training data:", X_train_processed.shape)
print("Testing data:", X_test_processed.shape)
print("Missing values:", np.isnan(X_train_processed.toarray()).sum()
      if hasattr(X_train_processed, "toarray") else np.isnan(X_train_processed).sum())

Training data: (5634, 46)
Testing data: (1409, 46)
Missing values: 0
